# 策略增强版 + 事件驱动执行仿真

接 `strategies.ipynb`（基础版）。本 notebook 同样**只做可视化**，结果由以下命令生成：

```bash
uv run python -m quant.data       # 下载历史成分股数据（默认股票池 hs300_pit）
uv run python -m quant.run        # 基础版 + 增强版（vectorbt 向量化回测）
uv run python -m quant.run_exec   # 增强版目标权重 -> backtrader 事件驱动执行仿真
```

三部分内容：
1. **股票池**：从"当前上证50成分股"换成**历史成分股（point-in-time）**，消除幸存者偏差
2. **增强版策略**：每个策略都与基础版同口径对比，并做消融（去掉某项改进看效果）
3. **执行仿真**：用 backtrader 重放目标权重，建模佣金/印花税/价差/冲击/参与率/涨跌停/停牌/整手，
   看"纸面收益"到"可实现收益"之间丢了多少，以及策略能容纳多大资金

In [ ]:
import os
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

from quant.viz import (BENCH, COST_COLORS, DIVERGING, GRID, INK2, ORDINAL, SERIES, plot_equity, plot_grid,
                       show_stats)

UNIVERSE = os.environ.get("QUANT_UNIVERSE", "cap300")  # 与 quant.config 一致
ROOT = Path("..") / "data" / "results"
def load(name: str, universe: str = UNIVERSE) -> dict:
    return pd.read_pickle(ROOT / universe / f"{name}.pkl")
print("股票池:", UNIVERSE)

## 0. 股票池与幸存者偏差

基础版用的是**今天的**上证50成分股去回测 2017 年以来的历史——这些股票之所以今天还在指数里，
恰恰因为它们过去涨得好（或至少没跌出去）。这会系统性地高估所有"买股票"策略的收益，
最明显的症状是：**"股票池等权"这个最傻的策略在基础版里表现最好**。

改进做法（`quant.data`）：每个月末向 baostock 查询**当时的**指数成分（`query_hs300_stocks(date=...)`），
下载样本期内所有曾经入选的股票（含已退市），构建 `member` 掩码：
某只股票只有在当时是成分股的日子才能被选入（`tradable = 未停牌 & 非ST & member`），
被剔出成分后仍可以卖出（`can_trade`）。

下表对比同一套基础版策略在两个股票池上的结果（需要两个股票池都运行过 `quant.run`）。

In [ ]:
if (ROOT / "sz50").exists() and UNIVERSE != "sz50":
    both = pd.concat({"sz50 当前成分(有偏)": load("summary", "sz50")["stats"],
                      f"{UNIVERSE} 历史成分": load("summary")["stats"]})
    cols = ["年化收益", "夏普比率", "最大回撤"]
    display(show_stats(both[cols].unstack(0).swaplevel(axis=1).sort_index(axis=1)))
else:
    print("只有一个股票池的结果。可分别用 QUANT_UNIVERSE=sz50 / hs300_pit 运行 quant.run 以对比。")

## 1. 动量·增强版

| 改进 | 动机 |
|---|---|
| 风险调整动量（收益/波动），合成 12-1 与 6-1 两个期限 | 高波动股票靠"运气"冲到榜首、A股高波动股票后续收益差；减少对单一回看期的依赖 |
| 缓冲区选股（前 N+5 名内不换） | 排名在边界附近来回跳动造成无效换手 |
| 逆波动率加权 + 单票上限 20% | 等权在高波动股票上承担了过多风险 |
| 组合波动率目标 20%（两遍法，无未来函数） | 动量崩溃通常发生在高波动的市场反转期 |

代码：`src/quant/enhanced/momentum.py`

In [ ]:
mom = load("momentum_plus")
show_stats(mom["stats"])

In [ ]:
plot_equity(mom, "动量+ vs 基础版")
s = mom["exposure"]
fig = go.Figure(go.Scatter(x=s.index, y=s, line=dict(color=SERIES[0], width=2, shape="hv"), hovertemplate="%{y:.0%}"))
fig.update_layout(title="波动率目标下的总仓位（调仓日）", yaxis_tickformat=".0%", height=280, yaxis_range=[0, 1.05])
fig.show()

## 2. 均值回归·增强版

基础版每只股票独立分配 1/N 资金、绝大部分时间空仓，并在趋势性下跌中反复接飞刀。

| 改进 | 动机 |
|---|---|
| 个股趋势过滤：收盘 > MA250 才买回调 | 上升趋势中的超跌更可能回归 |
| **大盘过滤：沪深300 > MA250 才开新仓** | 熊市里超跌信号成片出现、几乎都会继续下跌；扩到 300 只股票后这是最关键的修正 |
| **入场阈值 k=2.5** | 股票池越大，k=2 的信号越多、质量越差 |
| 共享资金池 + 20 个名额（每只 5%） | 提高资金利用率，同时分散 |
| 时间止损（30 天）、10% 止损、剔出股票池即卖 | 回归失败的仓位不无限期占用名额 |

> **过拟合的反面教材**：这套策略最初在上证50 上调参（k=2、10 名额、无大盘过滤），夏普 0.67；
> 原封不动搬到 cap300 后年化 −5.6%、最大回撤 −56%。修正后回撤大幅下降，但大盘股上的抄底型均值回归收益依然很低。

代码：`src/quant/enhanced/mean_reversion.py`

In [ ]:
mr = load("mean_reversion_plus")
show_stats(mr["stats"])

In [ ]:
plot_equity(mr, "均值回归+ vs 基础版")
plot_grid(mr["grid"], "夏普比率", "大盘过滤窗口（0=不过滤） × 入场阈值 k：夏普比率")

## 3. 趋势跟随·增强版

基础版只有一组参数、要么满仓要么空仓；参数网格本身说明单组参数极不稳定。

| 改进 | 动机 |
|---|---|
| 信号集成：3 组均线 + 3 组唐奇安 + 3 个时间序列动量，看多比例 = 仓位 | 不押注单一参数，仓位随趋势强度渐变 |
| 波动率目标 15% | 下跌常伴随波动放大，自动降仓 |
| 仓位变化 > 10% 才调整 | 避免每日微调 |

> 诚实地说：2017 年以来沪深300 整体震荡，**趋势择时在夏普上仍未跑赢买入持有**；增强版的价值主要是大幅降低回撤。
> 另外指数为价格指数（不含分红），空仓期现金不计利息，二者都低估了择时策略。

代码：`src/quant/enhanced/trend.py`

In [ ]:
tr = load("trend_plus")
show_stats(tr["stats"])

In [ ]:
plot_equity(tr, "趋势+ vs 基础版")
p, e = tr["price"], tr["exposure"]
fig = go.Figure()
fig.add_scatter(x=e.index, y=e, name="仓位", line=dict(color=SERIES[0], width=1.5, shape="hv"), fill="tozeroy",
                fillcolor="rgba(42,120,214,0.15)", yaxis="y2", hovertemplate="%{y:.0%}")
fig.add_scatter(x=p.index, y=p, name="沪深300", line=dict(color=INK2, width=1.5))
fig.update_layout(title="沪深300 与趋势+ 仓位", height=420,
                  yaxis2=dict(overlaying="y", side="right", tickformat=".0%", range=[0, 3], showgrid=False))
fig.show()
show_stats(tr["singles"][["年化收益", "夏普比率", "最大回撤"]])

## 4. 因子投资·增强版

| 改进 | 动机 |
|---|---|
| 滚动 24 个月 IC 加权（只用已实现的 IC，负值截 0），动量也作为候选 | 因子有效性随时间变化；在 cap300 上"小市值"IC 为负（大盘股里反而越大越好），IC 加权会自动把它降为 0 |
| **持仓 30 只 + 缓冲区（前 45 名不换）+ 逆波动率加权** | 降低个股噪声、换手与集中度——这是 cap300 上改进的主要来源 |
| 市值中性化（可选，默认关） | 市值前 300 的市值区间本就很窄，中性化没有帮助（持 10 只时还明显拖累），保留为对照列 |

代码：`src/quant/enhanced/factor.py`

In [ ]:
fac = load("factor_plus")
display(show_stats(fac["stats"]))
fac["composite_ic"].rename("复合因子 Rank IC 均值").to_frame().style.format("{:.4f}")

In [ ]:
plot_equity(fac, "因子+ vs 基础版")
fw = fac["factor_weights"]
fig = go.Figure()
for i, c in enumerate(fw.columns):
    fig.add_scatter(x=fw.index, y=fw[c], name=c, stackgroup="w", line=dict(width=0.5, color=SERIES[i]),
                    hovertemplate="%{y:.0%}")
fig.update_layout(title="滚动 IC 加权：各因子权重随时间变化", yaxis_tickformat=".0%", height=380)
fig.show()

In [ ]:
show_stats(load("summary_plus")["stats"])

## 5. 事件驱动执行仿真（backtrader）

向量化回测把执行简化成"次日开盘全部成交、单边 0.2% 成本"。实际执行中：大单会推动价格、
成交量有限只能分几天完成、涨跌停和停牌时根本成交不了、只能按整手下单、印花税只收卖方……

这里用成熟的事件驱动框架 **backtrader** 重放增强版策略的目标权重（`src/quant/execution.py`），
只通过它的标准扩展点注入 A 股规则：

| backtrader 扩展点 | 建模内容 |
|---|---|
| `PandasData` 附加 lines | 成交额、20 日 ADV、20 日波动（均滞后一天）、能否成交、一手对应的复权单位 |
| broker `filler` | 停牌不成交；开盘涨停买不进、跌停卖不出；单日成交 ≤ 当日成交额 × 10%；整手 |
| `CommInfoBase` | 佣金万 2.5（最低 5 元）+ 过户费 + 印花税（2023-08-28 起 0.1%→0.05%）+ 半个价差 + 平方根冲击 |
| `Strategy` | 信号日收盘按目标权重下单（先卖后买），未成交部分有效期 5 个交易日 |

市场冲击采用经验上广泛使用的平方根模型：$\text{impact} = \eta\,\sigma_{daily}\sqrt{Q/\text{ADV}}$，默认 $\eta=0.7$。
半价差 = max(0.005 元 / 实际股价, 2bp)，低价股相对价差更大。

In [ ]:
exe = load("execution")
rec = exe["reconcile"]
fig = go.Figure()
for i, c in enumerate(rec.columns):
    fig.add_scatter(x=rec.index, y=rec[c], name=c, line=dict(color=SERIES[i], width=2, dash="dot" if i else None))
fig.update_layout(title="对账：同一组权重，零摩擦 backtrader vs 零成本 vectorbt（应基本重合）", yaxis_type="log", height=360)
fig.show()

### 5.1 现实执行 vs 向量化回测（资金 1000 万）

上表为事件驱动执行结果；成本列为**年化成本占平均资产的基点数**。

In [ ]:
cols = ["年化收益", "夏普比率", "最大回撤", "总成本(年化bp)", "年化双边换手", "平均参与率", "受阻撮合次数", "过期/拒绝订单"]
display(show_stats(exe["real_stats"][cols]))
cmp = pd.DataFrame({"向量化 年化收益": exe["vector_stats"]["年化收益"], "事件驱动 年化收益": exe["real_stats"]["年化收益"]})
cmp["差异"] = cmp.iloc[:, 1] - cmp.iloc[:, 0]
cmp.style.format("{:.2%}")

In [ ]:
eq = exe["real_equity"]
fig = go.Figure()
for i, name in enumerate(exe["real_stats"].index):
    for c in [c for c in eq.columns if c.startswith(name + " ")]:
        real = "事件驱动" in c
        fig.add_scatter(x=eq.index, y=eq[c], name=c, line=dict(color=SERIES[i], width=2 if real else 1.2, dash=None if real else "dot"))
fig.update_layout(title="事件驱动执行（实线） vs 向量化回测（虚线）", yaxis_type="log", height=480)
fig.show()

parts = ["佣金", "过户费", "印花税", "价差", "冲击"]
rs = exe["real_stats"]
fig = go.Figure()
for p in parts:
    fig.add_bar(y=rs.index, x=rs[f"{p}(年化bp)"], name=p, orientation="h", marker_color=COST_COLORS[p],
                hovertemplate=f"{p} %{{x:.1f}}bp<extra></extra>")
fig.update_layout(barmode="stack", title="年化交易成本分解（bp / 年，1000 万资金）", height=300, hovermode="y unified",
                  xaxis_title="bp / 年")
fig.show()
exe["blocked"].style.format("{:.0f}")

### 5.2 资金容量

冲击成本随单笔规模的平方根增长，参与率上限则让大资金需要多天才能建完仓。
随着资金规模上升，**冲击成本**和**未成交（跟踪误差）**逐渐吞噬收益——这条曲线的拐点就是策略容量。

In [ ]:
cap = exe["capacity"].reset_index()
fig = go.Figure()
for i, (name, g) in enumerate(cap.groupby("策略", sort=False)):
    fig.add_scatter(x=g["资金"], y=g["年化收益"], name=name, mode="lines+markers", line=dict(color=SERIES[i], width=2),
                    customdata=g["冲击(年化bp)"], hovertemplate="资金 %{x:.2s}<br>年化 %{y:.1%}<br>冲击 %{customdata:.0f}bp<extra>" + name + "</extra>")
fig.update_layout(title="资金规模 vs 年化收益（事件驱动执行）", xaxis_type="log", xaxis_title="初始资金（元）",
                  yaxis_tickformat=".0%", hovermode="closest", height=400)
fig.show()
show_stats(exe["capacity"][["年化收益", "夏普比率", "冲击(年化bp)", "总成本(年化bp)", "平均参与率", "受阻撮合次数"]])

In [ ]:
cash = exe["sensitivity_cash"]
plot_grid(exe["sensitivity"], "年化收益", f"动量+ 年化收益：参与率上限 × 冲击系数（资金 {cash/1e8:.0f} 亿）", fmt=".1%")

### 5.3 换手控制的价值

基础版动量每月全量重排 Top10；增强版用缓冲区。在小资金下二者差别主要是信号质量，
资金越大，换手带来的冲击成本差异越明显。

In [ ]:
show_stats(exe["turnover_value"][["年化收益", "夏普比率", "总成本(年化bp)", "冲击(年化bp)", "年化双边换手"]])

## 小结与局限

- **幸存者偏差**是基础版最大的问题，换成历史成分股后所有"选股"策略的收益都明显下降，"股票池等权"不再是神话。
- **增强版**的改进都做了消融：真正起作用的是"风险调整 + 降换手"（动量）、"趋势过滤"（均值回归）、
  "降回撤"（趋势）、"中性化 + 组合构建"（因子）；单纯 IC 加权这类看起来更聪明的做法未必有效。
- **执行仿真**显示固定 0.2% 的假设在小资金下偏保守、在大资金下严重偏乐观；冲击成本决定了策略容量。

仍未建模：盘中价格路径（用开盘价近似成交）、分红税、融券做空、指数 ETF 的折溢价、
退市股票的退市整理期价格（停牌后按最后价格估值）。参数选择仍在同一段样本内完成，存在一定的过拟合风险，
严格的做法应保留样本外区间或做滚动前推检验。